# NumPy 实现 MNIST 前馈全连接神经网络

In [2]:
import numpy as np
from tensorflow.keras.datasets import mnist

np.random.seed(42)

In [ ]:
#加载并预处理 MNIST 数据
def to_one_hot(y, num_classes=10):
    one_hot = np.zeros((len(y), num_classes))
    one_hot[np.arange(len(y)), y] = 1.0
    return one_hot


(x_train, y_train_raw), (x_test, y_test_raw) = mnist.load_data()

train_size = 10000
test_size = 2000

x_train = x_train[:train_size].reshape(train_size, 784).astype(np.float64) / 255.0
y_train = to_one_hot(y_train_raw[:train_size], 10)
y_train_raw = y_train_raw[:train_size]

x_test = x_test[:test_size].reshape(test_size, 784).astype(np.float64) / 255.0
y_test = to_one_hot(y_test_raw[:test_size], 10)
y_test_raw = y_test_raw[:test_size]

print('x_train:', x_train.shape)
print('y_train:', y_train.shape)
print('x_test:', x_test.shape)
print('y_test:', y_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
x_train: (10000, 784)
y_train: (10000, 10)
x_test: (2000, 784)
y_test: (2000, 10)


In [ ]:
#定义网络所需函数
def relu(x):
    return np.maximum(0, x)


def relu_grad(x):
    return (x > 0).astype(np.float64)


def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


def cross_entropy_loss(probs, y_one_hot):
    eps = 1e-12
    return -np.mean(np.sum(y_one_hot * np.log(probs + eps), axis=1))


def accuracy(probs, y_true):
    pred = np.argmax(probs, axis=1)
    return np.mean(pred == y_true)

In [ ]:
#初始化参数
input_dim = 784
hidden_dim = 128
output_dim = 10

W1 = 0.01 * np.random.randn(input_dim, hidden_dim)
b1 = np.zeros((1, hidden_dim))
W2 = 0.01 * np.random.randn(hidden_dim, output_dim)
b2 = np.zeros((1, output_dim))

In [ ]:
# 前向传播与反向传播
def forward_pass(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    probs = softmax(z2)
    cache = {
        'X': X,
        'z1': z1,
        'a1': a1,
        'probs': probs
    }
    return probs, cache


def backward_pass(cache, Y, W2):
    X = cache['X']
    z1 = cache['z1']
    a1 = cache['a1']
    probs = cache['probs']
    batch_size = X.shape[0]

    dz2 = (probs - Y) / batch_size
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    da1 = dz2 @ W2.T
    dz1 = da1 * relu_grad(z1)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    return dW1, db1, dW2, db2

## 5. 训练模型

In [8]:
def iterate_minibatches(X, Y, batch_size=128, shuffle=True):
    indices = np.arange(X.shape[0])
    if shuffle:
        np.random.shuffle(indices)
    for start_idx in range(0, X.shape[0], batch_size):
        batch_idx = indices[start_idx:start_idx + batch_size]
        yield X[batch_idx], Y[batch_idx]


learning_rate = 0.1
epochs = 20
batch_size = 128

for epoch in range(epochs):
    for batch_x, batch_y in iterate_minibatches(x_train, y_train, batch_size=batch_size):
        probs, cache = forward_pass(batch_x, W1, b1, W2, b2)
        dW1, db1, dW2, db2 = backward_pass(cache, batch_y, W2)

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

    train_probs, _ = forward_pass(x_train, W1, b1, W2, b2)
    test_probs, _ = forward_pass(x_test, W1, b1, W2, b2)

    train_loss = cross_entropy_loss(train_probs, y_train)
    train_acc = accuracy(train_probs, y_train_raw)
    test_acc = accuracy(test_probs, y_test_raw)

    print(f'epoch={epoch + 1:02d} loss={train_loss:.4f} train_acc={train_acc:.4f} test_acc={test_acc:.4f}')

epoch=01 loss=1.3089 train_acc=0.6359 test_acc=0.5930
epoch=02 loss=0.6221 train_acc=0.8091 test_acc=0.7775
epoch=03 loss=0.4713 train_acc=0.8677 test_acc=0.8320
epoch=04 loss=0.3783 train_acc=0.8983 test_acc=0.8625
epoch=05 loss=0.3544 train_acc=0.8996 test_acc=0.8645
epoch=06 loss=0.3315 train_acc=0.9033 test_acc=0.8655
epoch=07 loss=0.3039 train_acc=0.9132 test_acc=0.8805
epoch=08 loss=0.3203 train_acc=0.9029 test_acc=0.8660
epoch=09 loss=0.3164 train_acc=0.9062 test_acc=0.8640
epoch=10 loss=0.2600 train_acc=0.9285 test_acc=0.8865
epoch=11 loss=0.2531 train_acc=0.9310 test_acc=0.8870
epoch=12 loss=0.3748 train_acc=0.8819 test_acc=0.8395
epoch=13 loss=0.2382 train_acc=0.9316 test_acc=0.8930
epoch=14 loss=0.2282 train_acc=0.9344 test_acc=0.8955
epoch=15 loss=0.2238 train_acc=0.9343 test_acc=0.8885
epoch=16 loss=0.2221 train_acc=0.9337 test_acc=0.8840
epoch=17 loss=0.2082 train_acc=0.9387 test_acc=0.8950
epoch=18 loss=0.1961 train_acc=0.9442 test_acc=0.8960
epoch=19 loss=0.1795 train_a

## 6. 测试集评估

In [9]:
test_probs, _ = forward_pass(x_test, W1, b1, W2, b2)
test_pred = np.argmax(test_probs, axis=1)
test_acc = np.mean(test_pred == y_test_raw)
print('Final test accuracy:', test_acc)

Final test accuracy: 0.8885
